# Run FunSearch on Bin Packing
Five steps:
1. Implement 'LLM' interface.
2. Implement a 'SandBox' interface.
3. Prepare a 'specification'.
4. Prepare a dataset.
5. Start FunSearch.

In [8]:
!pip install openjij
!pip install pyqubo

In [19]:
!git clone https://github.com/lhj-7777777/SQA-TSP.git

import sys

sys.path.append('/content/SQA-TSP')

fatal: destination path 'SQA-TSP' already exists and is not an empty directory.


In [21]:
sys.path.append('/content/SQA-TSP')

In [23]:
import sys
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/SQA-TSP/', '/content/SQA-TSP/', '/content/SQA-TSP/', '/content/SQA-TSP/', '/content/SQA-TSP', '/content/SQA-TSP']


## Preparation: download the project file from github. And update system path.

## 1. Implement LLM interface
Set the API's IP address according to your API provider (See line 65 in the following code).
```python
conn = http.client.HTTPSConnection("api.chatanywhere.com.cn")
```
You should prepare a 'key' for the LLM API. And fill them in the header (See line 76-80 in the following code).
```python
headers = {
    'Authorization': 'Bearer [put your key here, the key may start with "sk-..."]',
    'User-Agent': 'Apifox/1.0.0 (https://apifox.com)',
    'Content-Type': 'application/json'
}
```

In [24]:
import time
import json
import multiprocessing
from typing import Collection, Any
import http.client
from implementation import sampler


def _trim_preface_of_body(sample: str) -> str:
    """Trim the redundant descriptions/symbols/'def' declaration before the function body.
    Please see my comments in sampler.LLM (in sampler.py).
    Since the LLM used in this file is not a pure code completion LLM, this trim function is required.

    -Example sample (function & description generated by LLM):
    -------------------------------------
    This is the optimized function ...
    def priority_v2(...) -> ...:
        return ...
    This function aims to ...
    -------------------------------------
    -This function removes the description above the function's signature, and the function's signature.
    -The indent of the code is preserved.
    -Return of this function:
    -------------------------------------
        return ...
    This function aims to ...
    -------------------------------------
    """
    lines = sample.splitlines()
    func_body_lineno = 0
    find_def_declaration = False
    for lineno, line in enumerate(lines):
        # find the first 'def' statement in the given code
        if line[:3] == 'def':
            func_body_lineno = lineno
            find_def_declaration = True
            break
    if find_def_declaration:
        code = ''
        for line in lines[func_body_lineno + 1:]:
            code += line + '\n'
        return code
    return sample


class LLMAPI(sampler.LLM):
    """Language model that predicts continuation of provided source code.
    """

    def __init__(self, samples_per_prompt: int, trim=True):
        super().__init__(samples_per_prompt)
        additional_prompt = ('Complete a different and more complex Python function. '
                             'Be creative and you can insert multiple if-else and for-loop in the code logic.'
                             'Only output the Python code, no descriptions.')
        self._additional_prompt = additional_prompt
        self._trim = trim

    def draw_samples(self, prompt: str) -> Collection[str]:
        """Returns multiple predicted continuations of `prompt`."""
        return [self._draw_sample(prompt) for _ in range(self._samples_per_prompt)]

    def _draw_sample(self, content: str) -> str:
        prompt = '\n'.join([content, self._additional_prompt])
        while True:
            try:
                conn = http.client.HTTPSConnection("api.chatanywhere.com.cn")
                payload = json.dumps({
                    "max_tokens": 512,
                    "model": "gpt-3.5-turbo",
                    "messages": [
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ]
                })
                headers = {
                    'Authorization': 'Bearer sk-ys02zx......(replace with your own)......',
                    'User-Agent': 'Apifox/1.0.0 (https://apifox.com)',
                    'Content-Type': 'application/json'
                }
                conn.request("POST", "/v1/chat/completions", payload, headers)
                res = conn.getresponse()
                data = res.read().decode("utf-8")
                data = json.loads(data)
                response = data['choices'][0]['message']['content']
                # trim function
                if self._trim:
                    response = _trim_preface_of_body(response)
                return response
            except Exception:
                time.sleep(2)
                continue

## 2. Implement a 'SandBox' interface

In [25]:
from implementation import evaluator
from implementation import evaluator_accelerate


class Sandbox(evaluator.Sandbox):
    """Sandbox for executing generated code. Implemented by RZ.

    RZ: Sandbox returns the 'score' of the program and:
    1) avoids the generated code to be harmful (accessing the internet, take up too much RAM).
    2) stops the execution of the code in time (avoid endless loop).
    """

    def __init__(self, verbose=False, numba_accelerate=True):
        """
        Args:
            verbose         : Print evaluate information.
            numba_accelerate: Use numba to accelerate the evaluation. It should be noted that not all numpy functions
                              support numba acceleration, such as np.piecewise().
        """
        self._verbose = verbose
        self._numba_accelerate = numba_accelerate

    def run(
            self,
            program: str,
            function_to_run: str,  # RZ: refers to the name of the function to run (e.g., 'evaluate')
            function_to_evolve: str,  # RZ: accelerate the code by decorating @numba.jit() on function_to_evolve.
            inputs: Any,  # refers to the dataset
            test_input: str,  # refers to the current instance
            timeout_seconds: int,
            **kwargs  # RZ: add this
    ) -> tuple[Any, bool]:
        """Returns `function_to_run(test_input)` and whether execution succeeded.

        RZ: If the generated code (generated by LLM) is executed successfully,
        the output of this function is the score of a given program.
        RZ: PLEASE NOTE THAT this SandBox is only designed for bin-packing problem.
        """
        dataset = inputs[test_input]
        try:
            result_queue = multiprocessing.Queue()
            process = multiprocessing.Process(
                target=self._compile_and_run_function,
                args=(program, function_to_run, function_to_evolve, dataset, self._numba_accelerate, result_queue)
            )
            process.start()
            process.join(timeout=timeout_seconds)
            if process.is_alive():
                # if the process is not finished in time, we consider the program illegal
                process.terminate()
                process.join()
                results = None, False
            else:
                if not result_queue.empty():
                    results = result_queue.get_nowait()
                else:
                    results = None, False

            return results
        except:
            return None, False

    def _compile_and_run_function(self, program, function_to_run, function_to_evolve, dataset, numba_accelerate,
                                  result_queue):
        try:
            # optimize the code (decorate function_to_run with @numba.jit())
            if numba_accelerate:
                program = evaluator_accelerate.add_numba_decorator(
                    program=program,
                    function_to_evolve=function_to_evolve
                )
            # compile the program, and maps the global func/var/class name to its address
            all_globals_namespace = {}
            # execute the program, map func/var/class to global namespace
            exec(program, all_globals_namespace)
            # get the pointer of 'function_to_run'
            function_to_run = all_globals_namespace[function_to_run]
            # return the execution results
            results = function_to_run(dataset)
            # the results must be int or float
            if not isinstance(results, (int, float)):
                result_queue.put((None, False))
                return
            result_queue.put((results, True))
        except Exception:
            # if raise any exception, we assume the execution failed
            result_queue.put((None, False))

## 3. Prepare a 'specification'

In [ ]:
specification = r'''
import itertools
import numpy as np
import funsearch
from scipy.spatial.distance import cdist
from pyqubo import Constraint,Array
import time
from openjij import SQASampler

locate = np.array([
    [1, 320, 100],
    [2, 340, 150],
    [3, 180, 190],
    [4, 160, 130],
    [5, 250, 170],
    [6, 250, 80],
    [7, 330, 200],
    [8, 220, 220]
])
coords = locate[:, 1:]
n_pos = len(locate)

L_original = cdist(coords, coords, metric='euclidean')
L = L_original / L_original.max()


@funsearch.run
def evaluate(n=None) -> int:
    """
    Return the negative score of the path length (1000 - path length); maximizing this value is equivalent to minimizing the path length
    n: Parameter for interface compatibility (the global n_pos is used in practice)
    """
    _, path_length = solve()
    score = 1000 - path_length
    return score


def solve() -> tuple:
    """
    Construct QUBO model → Solve QUBO using SQASampler → Parse the path (screen with Hbind3 among objects that satisfy Hbind1 and Hbind2 constraints) → Calculate length
    """
    X = Array.create("X", shape=(n_pos, n_pos), vartype="BINARY")
    X_np = np.array(X).reshape(n_pos, n_pos)

    Xtmp = np.concatenate([X_np, X_np[:, 0].reshape(-1, 1)], axis=1)
    Y = np.delete(Xtmp, 0, 1)

    # Penalty coefficients
    a1 = 500.0  # Penalty for visiting only one node at each position
    a2 = 500.0  # Penalty for visiting each node only once
    a3 = 100.0  # Coefficient of the evolutionary term

    # Construct objective function Hobj (minimize path length)
    Hobj = np.sum(X_np @ Y.T * L)

    # Constraint 1: Only one node i can be selected for each position j; the result must not violate Constraint 1
    Hbind1 = np.sum((1 - np.sum(X_np, axis=0)) ** 2)

    # Constraint 2: Each node i can be selected only once; the result must not violate Constraint 2
    Hbind2 = np.sum((1 - np.sum(X_np, axis=1)) ** 2)

    # Constraint 3: Call the evolved priority function to generate Hbind3 (initially 0), which does not conflict with Constraint 1 and Constraint 2
    Hbind3 = priority(X, Y, L, n_pos)

    # Total Hamiltonian (objective function + penalty terms)
    H = Hobj + a1 * Hbind1 + a2 * Hbind2 + a3 * Hbind3

    # Compile QUBO model
    model = H.compile()
    qubo, offset = model.to_qubo()

    sampler = SQASampler()

    response = sampler.sample_qubo(
        qubo, schedule=None, beta=5.0, trotter=8, gamma=1.0,
        num_sweeps=10,
        num_reads=10
    )
    # Get the optimal solution
    result = min(response.record, key=lambda x: x[1])
    X1 = result[0].reshape((n_pos, n_pos))
    # Node number corresponding to each location
    circuit = np.argmax(X1, axis=0).tolist()
    # Return to the starting point
    circuit.append(circuit[0])

    # Calculate the actual path length (using the original distance matrix)
    path_length = 0.0
    for i in range(len(circuit)-1):
        u = circuit[i]
        v = circuit[i+1]
        path_length += L_original[u][v]

    return circuit, path_length


@funsearch.evolve
def priority(X, Y, L, n_pos) -> Constraint:
    """
    FunSearch evolutionary function: Generate constraint term Hbind3
    Parameter description:
    - X_np: PyQubo's Binary matrix (X[i,j] indicates whether node i is selected at position j)
    - Y: Cyclically shifted X matrix (Y[i,j] = X[i,(j+1)%n_pos])
    - L: Normalized distance matrix
    - n_pos: Number of nodes
    - a1/a2/a3: Penalty coefficients (can be referenced during evolution)
    - Do not violate the two constraints Hbind1 and Hbind2; the final circuit satisfies Constraint 1 and Constraint 2
    - Screen with Hbind3 among objects that satisfy Hbind1 and Hbind2 constraints
    Return: PyQubo's Constraint object (or symbolic expression)
    """
    return 0.0
'''

## 4. Prepare a dataset

In [ ]:
#import bin_packing_utils

#bin_packing_or3 = {'OR3': bin_packing_utils.datasets['OR3']}

## 5. Start FunSearch
Please note that in jupyter notebook the following code will fail. This is because juypter does not support multiprocessing. Colab backend supports multiprocessing.

In [ ]:
from implementation import funsearch
from implementation import config

# It should be noted that the if __name__ == '__main__' is required.
# Because the inner code uses multiprocess evaluation.
if __name__ == '__main__':
    class_config = config.ClassConfig(llm_class=LLMAPI, sandbox_class=Sandbox)
    config = config.Config(samples_per_prompt=4, evaluate_timeout_seconds=30)
    global_max_sample_num = 10  # if it is set to None, funsearch will execute an endless loop
    funsearch.main(
        specification=specification,
        inputs=bin_packing_or3,
        config=config,
        max_sample_nums=global_max_sample_num,
        class_config=class_config,
        log_dir='../logs/funsearch_llm_api'
    )